In [5]:
# Password-protected file access (to be used across Jupyter notebooks)

import msoffcrypto # To access password-protected files
import pandas as pd # To read and manipulate data
from io import BytesIO # Temp handling of decrypted files
from getpass import getpass # Get password input w/out encoding

password = getpass("Enter anon file password: ") # Prompt for password
decrypted_file = BytesIO() # Creates temp file in memory

with open(r"C:\Users\karin\Documents\2. Data\Anonymous_Data.xlsx", "rb") as f: # Open the password-protected file
    office_file = msoffcrypto.OfficeFile(f) # Create Officefile object
    office_file.load_key(password=password) # Load password
    office_file.decrypt(decrypted_file) # Decrypt file into memory

df = pd.read_excel(decrypted_file) # Read decrypted file into a DataFrame (DF)
df.shape # Display the shape of the DF (rows and columns)

(2839, 88)

In [6]:
# Rebuild df_binary for attendance feature below

fail_categories = ['Fail Resit', 'Fail Withdraw', 'Repeat without Attendance', 'Repeat with Attendance', 'Complete Repeat'] # Defining categories which count as a fail

df_binary = df[df['Progression Decision'] != 'Trail Progress'].copy() # Create a new DF excluding 'Trail Progress' rows as it is its own edge case

df_binary['initially_failed'] = df_binary['Progression Decision'].isin(fail_categories) # Create new column in new DF: True if student failed and false if they passed

df_binary.shape # Display the shape of the DF (rows and columns)

(2837, 89)

In [7]:
# Attendance Feature: already numeric but accouting for off-site students with null values

df_binary['is_offsite'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary['is_offsite'].value_counts() # Count the number of off-site students (True) and on-site students (False)

is_offsite
False    2785
True       52
Name: count, dtype: int64

Checked the data and realised there was a slight error with column alignment and matching student ID's. Corrected it which enabled further distinction of this number to split by administrative student status details.

In [ ]:
# Attendance Feature: check if any off-site students are categorised as on-site via admin student status

df_binary['no_attendance_data'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary[df_binary['no_attendance_data']]['Student Status'].value_counts() # Count no. students off-site but categorised via admin student status

Student Status
Off-site     41
PR/Repeat    10
Normal        1
Name: count, dtype: int64

Off-site we have 41 students (expected as 41 were off-site therefore did not have their attendance data recorded), then 10 who were repeat students, presumably with no attendance requirement hence no data. The one normal is an unexplained case.

In [ ]:
# Breakdown of student numbers via three student statuses

df_binary['Off-site'] = df_binary['Student Status'].str.contains('Off-site', case=False, na=False) # Create new column which turns true if student status contains 'Off-site' (off-site student)
df_binary['Repeating'] = df_binary['Student Status'].str.contains('PR/Repeat', case=False, na=False) # Create new column which turns true if student status contains 'PR/Repeat' (repeating student)
df_binary['Unexplained Null Attendance'] = df_binary['no_attendance_data'] & ~df_binary['Off-site'] & ~df_binary['Repeating'] # Create new column which turns true if student has no attendance data but is not off-site or repeating

df_binary[['Off-site', 'Repeating', 'Unexplained Null Attendance']].sum() # Count students in each category

Off-site                       42
Repeating                      92
Unexplained Null Attendance     1
dtype: int64

Administrative student status does not record this detail and manual adjustments were made during excel data cleaning to clarify the data. The additional off-site students was caught accidentally as they fell outside of the expected cohort of off-site students. Drastic increase in repeating students due to the split between those expected to repeat with attendance or to repeat without having to attend.

In [ ]:
#

df_